In [1]:
# silence warnings
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import datetime as dt
from passwords import *
import pyodbc

try:
    from surgeo import BIFSGModel
except ModuleNotFoundError:
    ! pip install surgeo
    from surgeo import BIFSGModel

try:
    from gender_guesser.detector import Detector
except ModuleNotFoundError:
    ! pip install gender_guesser
    from gender_guesser.detector import Detector

In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2025-01-30 16:54:19.624609


#### Functions

In [3]:
def remove_hyphen(str_value):
    try:
        return int(str_value.split('-')[0].split('.')[0])
    except ValueError:
        return 'ERROR'

In [4]:
def get_race(ser_race_prob):
    return ser_race_prob.idxmax()

In [5]:
def get_gender(str_first_name, cls_detector):
    # return
    return cls_detector.get_gender(str_first_name)

#### Constants

In [6]:
str_split = '\\'
# project
str_project = os.getcwd().split(str_split)[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split(str_split)[5]
print(f'Task: {str_task}')

str_dirname_output = './output'

Project: 20241112-simple-model-test
Task: disparate_targets


#### Make output directory

In [7]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Connect to DB

In [8]:
# connnect to db
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)

#### Pull into DF

In [9]:
# get query
str_filename = 'query.sql'
str_local_path = f'./sql/{str_filename}'
str_query = open(str_local_path, 'r').read()

# pull into df
df = pd.read_sql_query(
    str_query, 
    con=conn,
)
list_cols = [
    'bigAccountId',
    'bigDebtorId',
    'bitDebtor',
    'strNameFirst',
    'strNameLast',
    'strZipCode',
]
df = df[list_cols]

# show
df

,bigAccountId,bigDebtorId,bitDebtor,strNameFirst,strNameLast,strZipCode
0,6476061,8090461.0,1,Oneal,Tometi,85024
1,6476062,8090462.0,1,Eli,Barker,98499
2,6476063,8090463.0,1,Theresa,Laster-Savoy,21208
3,6476064,8090464.0,1,JONATHON,MOORE,76302
4,6476065,8090465.0,1,Elizabeth,Martinez,77396
...,...,...,...,...,...,...
3290838,7627125,9469600.0,1,SHALTONTAE,WELLS,34748
3290839,7627126,9469601.0,1,Dawn,Rhoads,21076
3290840,7627127,9469602.0,1,Chiquitq,Williams,47904
3290841,7627128,9469603.0,1,Antonyo,Gray,46210


#### Clean data

In [10]:
%%time

# convert zip to str
df['strZipCode'] = df['strZipCode'].astype(str)

# rm hyphen
df['strZipCode'] = df['strZipCode'].apply(remove_hyphen)

# filter errors
df = df[df['strZipCode'] != 'ERROR'].copy()

# set dtype
for col in ['bigAccountId','bigDebtorId','strZipCode']:
    df[col] = df[col].astype(int)

# capitalize names
for col in ['strNameFirst','strNameLast']:
    df[col] = df[col].str.upper()

Wall time: 4.47 s


#### Get race

In [11]:
%%time

# init
cls_detector = BIFSGModel()

# get probability of each race
list_cols = ['white','black','api','native','multiple','hispanic']
df[list_cols] = cls_detector.get_probabilities(
    df['strNameFirst'],
    df['strNameLast'],
    df['strZipCode'],
)[list_cols]

# get max prob
df['max_proba_race'] = df[list_cols].apply(max, axis=1)

# get race
df['race'] = df[list_cols].apply(get_race, axis=1)

# drop
df.drop(list_cols, axis=1, inplace=True)

# show
df

Wall time: 2min 52s


,bigAccountId,bigDebtorId,bitDebtor,strNameFirst,strNameLast,strZipCode,max_proba_race,race
0,6476061,8090461,1,ONEAL,TOMETI,85024,NaN,NaN
1,6476062,8090462,1,ELI,BARKER,98499,0.872785,white
2,6476063,8090463,1,THERESA,LASTER-SAVOY,21208,NaN,NaN
3,6476064,8090464,1,JONATHON,MOORE,76302,0.967576,white
4,6476065,8090465,1,ELIZABETH,MARTINEZ,77396,0.971855,hispanic
...,...,...,...,...,...,...,...,...
3290838,7627125,9469600,1,SHALTONTAE,WELLS,34748,NaN,NaN
3290839,7627126,9469601,1,DAWN,RHOADS,21076,NaN,NaN
3290840,7627127,9469602,1,CHIQUITQ,WILLIAMS,47904,NaN,NaN
3290841,7627128,9469603,1,ANTONYO,GRAY,46210,NaN,NaN


#### Get gender

In [12]:
%%time

# init
cls_detector = Detector()

# capitalize first name
df['strNameFirst_cap'] = df['strNameFirst'].str.capitalize()

# make target
df['gender'] = df.apply(
    lambda x: get_gender(
        str_first_name=x['strNameFirst_cap'], 
        cls_detector=cls_detector,
    ),
    axis=1,
)

# drop
df.drop('strNameFirst_cap', axis=1, inplace=True)

# show value counts
df['gender'].value_counts()

Wall time: 33.4 s


gender
male             1224002
female           1083566
unknown           715617
mostly_female     128170
mostly_male       113958
andy               25062
Name: count, dtype: int64

In [13]:
# subset
df = df[df['gender'].isin(['male','female','mostly_female','mostly_male'])].copy()

# show value counts
df['gender'].value_counts()

gender
male             1224002
female           1083566
mostly_female     128170
mostly_male       113958
Name: count, dtype: int64

In [14]:
# map
dict_map = {
    'male': 'male',
    'mostly_male': 'male',
    'female': 'female',
    'mostly_female': 'female',
}

# map target
df['gender'] = df['gender'].map(dict_map)

# show value counts
df['gender'].value_counts()

gender
male      1337960
female    1211736
Name: count, dtype: int64

#### Show

In [15]:
# show
df

,bigAccountId,bigDebtorId,bitDebtor,strNameFirst,strNameLast,strZipCode,max_proba_race,race,gender
0,6476061,8090461,1,ONEAL,TOMETI,85024,NaN,NaN,male
1,6476062,8090462,1,ELI,BARKER,98499,0.872785,white,female
2,6476063,8090463,1,THERESA,LASTER-SAVOY,21208,NaN,NaN,female
3,6476064,8090464,1,JONATHON,MOORE,76302,0.967576,white,male
4,6476065,8090465,1,ELIZABETH,MARTINEZ,77396,0.971855,hispanic,female
...,...,...,...,...,...,...,...,...,...
3290835,7627122,9469597,1,MISTY,HARDING,97213,NaN,NaN,female
3290836,7627123,9469598,1,PEDRO,SAMA LOZADA,24012,NaN,NaN,male
3290837,7627124,9469599,1,MARY,WICKS,63146,NaN,NaN,female
3290839,7627126,9469601,1,DAWN,RHOADS,21076,NaN,NaN,female


#### Save locally

In [16]:
%%time

str_filename = 'df.gzip'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_parquet(
    str_local_path,
    compression='gzip',
)

Wall time: 14.2 s
